In [1]:
import pandas as pd
import numpy as np 
import boto3
from sklearn.model_selection import train_test_split
import sagemaker
from sagemaker import Session
import io
import sagemaker.amazon.common as smac 
import os 
from sagemaker.amazon.amazon_estimator import get_image_uri

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


### Data Preparation

In [2]:
df = pd.read_csv('student_scores.csv')

In [3]:
df.head()

,Hours,Scores
0,2.5,21
1,5.1,47
2,3.2,27
3,8.5,75
4,3.5,30


In [4]:
df.shape

(25, 2)

In [5]:
# separate x and y
x = df[["Hours"]]
y = df[["Scores"]]

In [6]:
y = y.astype("float32")
x = x.astype("float32")

In [7]:
# split the data

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2)

In [8]:
# reset index

x_train = x_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
x_test = x_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [9]:
x_train.head()

,Hours
0,7.7
1,9.2
2,5.1
3,1.9
4,3.2


In [10]:
y_train = y_train.iloc[:,0]

In [11]:
y_test = y_test.iloc[:,0]

### Sagemaker Session

In [26]:
sagemaker_session = sagemaker.Session()
bucket_name = "ml-deployment13"
prefix="linear-reg"
role = sagemaker.get_execution_role()

### Model Training

In [27]:
x_train = np.array(x_train)

In [28]:
# create buffer
buf = io.BytesIO()
smac.write_numpy_to_dense_tensor(buf, x_train, y_train)
buf.seek(0)

0

In [29]:
import sagemaker
import boto3

sess = sagemaker.Session()
# sagemaker session bucket -> used for uploading data, models and logs
# sagemaker will automatically create this bucket if it not exists
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

## Role Management
# To execute any kind of code specifically using sagemaker always provide role access
try:
    role = sagemaker.get_execution_role()
except ValueError:
    iam = boto3.client('iam')
    role = iam.get_role(RoleName='sagemaker_execution_role')['Role']['Arn']

sess = sagemaker.Session(default_bucket=sagemaker_session_bucket)

print(f"sagemaker role arn: {role}")
print(f"sagemaker session region: {sess.boto_region_name}")
print(f"sagemaker bucket name: {sagemaker_session_bucket}")

sagemaker role arn: arn:aws:iam::594754932627:role/service-role/AmazonSageMaker-ExecutionRole-20251124T124771
sagemaker session region: us-east-1
sagemaker bucket name: sagemaker-us-east-1-594754932627


In [30]:
import io, os, boto3, sagemaker
import sagemaker.amazon.common as smac

sagemaker_session = sagemaker.Session()
bucket_name = "sagemaker-us-east-1-594754932627"
prefix = "linear-reg"
role = sagemaker.get_execution_role()

# create buffer and write
buf = io.BytesIO()
smac.write_numpy_to_dense_tensor(buf, x_train, y_train)

# extract bytes and create a fresh open BytesIO
data_bytes = buf.getvalue()
buf = io.BytesIO(data_bytes)
buf.seek(0)

key = "student-data"
s3_key = os.path.join(prefix, "train", key)

boto3.resource("s3").Bucket(bucket_name).Object(s3_key).upload_fileobj(buf)
s3_train_data = f"s3://{bucket_name}/{s3_key}"
print("data uploaded:", s3_train_data)


data uploaded: s3://sagemaker-us-east-1-594754932627/linear-reg/train/student-data


In [31]:
x_test = np.array(x_test)
buf = io.BytesIO()
smac.write_numpy_to_dense_tensor(buf,x_test,y_test) 
buf.seek(0)

key = "student-data-test"
s3_key = os.path.join(prefix, "test", key)
boto3.resource("s3").Bucket(bucket_name).Object(s3_key).upload_fileobj(buf)
s3_test_data = f"s3://{bucket_name}/{prefix}/test/{key}"
print("data uploaded:", s3_test_data)


data uploaded: s3://sagemaker-us-east-1-594754932627/linear-reg/test/student-data-test


In [32]:
output_location = f"s3://{bucket_name}/{prefix}/output"
output_location

's3://sagemaker-us-east-1-594754932627/linear-reg/output'

In [34]:
# bring the container
container = sagemaker.image_uris.retrieve("linear-learner",boto3.Session().region_name)

In [36]:
# define the estimator
linear = sagemaker.estimator.Estimator(container, role, instance_count=1, instance_type="ml.c4.xlarge", output_path=output_location, sagemaker_session=sagemaker_session)

In [37]:
# setting up the hyperparameter 
linear.set_hyperparameters(feature_dim=1, predictor_types="regressor", mini_batch_size=4, epochs=6, num_models=32,loss="absolute_loss")

In [40]:
# fit the model
linear.fit({"train":s3_train_data})

INFO:sagemaker:Creating training-job with name: linear-learner-2025-11-24-17-50-27-139


2025-11-24 17:50:30 Starting - Starting the training job...
2025-11-24 17:51:02 Downloading - Downloading input data...
2025-11-24 17:51:27 Downloading - Downloading the training image............
2025-11-24 17:53:18 Training - Training image download completed. Training in progress..Docker entrypoint called with argument(s): train
Running default environment configuration script
[11/24/2025 17:53:28 INFO 140321814845248] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 'init_method': 'uniform', 'init_scale': '0.07', 'init_sigma': '0.01', 'init_bias': '0.0', 'optimizer': 'auto', 'loss': 'auto', 'margin': '1.0', 'quantile': '0.5', 'loss_insensitivity': 

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 # fit the model                                                                              │
│ ❱ 2 linear.fit({"train":s3_train_data})                                                          │
│   3                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:167 in wrapper  │
│                                                                                                  │
│   164 │   │   │   │   │   caught_ex = e                                                          │
│   165 │   │   │   │   finally:                                                                   │
│   166 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 167 │   │   │   │   │   │   raise caught_ex                                                    │
│   168 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   169 │   │   │   else:                                                                          │
│   170 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:138 in wrapper  │
│                                                                                                  │
│   135 │   │   │   │   start_timer = perf_counter()                                               │
│   136 │   │   │   │   try:                                                                       │
│   137 │   │   │   │   │   # Call the original function                                           │
│ ❱ 138 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   139 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   140 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   141 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/estimator.py:1380 in fit                       │
│                                                                                                  │
│   1377 │   │   │   wait = True                             

In [ ]:
# deploy the model
linear_regressor = linear.deploy(initial_instance_count=1, instance_type = "ml.m4.xlarge")

In [ ]:
linear_regressor.serializer=sagemaker.serializers.CSVSerializer()
linear_regressor.deserializer=sagemaker.deserializers.JSONDeserializer()

In [ ]:
# prediction

results = linear_regressor.predict(x_test)
results

In [ ]:
predictions = np.array([i["sc